PySpark Sales Data Project
Dataset link: 

https://drive.google.com/file/d/1nDxlZq8zRpfJdVtUZ5qPOwufcI19_x8f/view?usp=sharing

Step 0: Upload sales.csv to Volume

Step 1: Read Data
Read sales.csv
Display the data
Check the schema
Count the total records

Step 2: Explore the Data
Find the total number of rows
Find the total number of columns
Find distinct customers
Find distinct products
Generate summary statistics

Step 3: Clean the Data
Remove duplicate rows
Remove duplicate records based on transaction_id
Remove rows containing null values

Step 4: Transform the Data
Rename columns
Add new derived columns
Convert order_date to DateType
Add metadata columns:
current_timestamp()
current_date()
file_name(), file_path() 

Step 5: Write the Data
Write the cleaned data as Delta
Create a Delta table from the cleaned data

Step 6: Analytics (SQL, PySpark)
 Using the cleaned Delta table, answer the following business questions:
Calculate the total sales.
Find the total number of orders.
Find the total number of unique customers.
Calculate total sales for each customer.
Calculate total sales for each product.
Display the Top 10 customers by sales.
Display the Top 10 products by sales.
Find the highest sales transaction.
Find the lowest sales transaction.
Calculate the average order value.

Store all the results in views 


In [0]:
%run "/Workspace/Users/rahulpatel@cyntexa.com/Data_Brick_projects/include"

In [0]:
# Step 0: Upload sales.csv to Volume
# Step 1: Read Data Read sales.csv Display the data Check the schema Count the total records
df = spark.read.csv("/Volumes/dev_cat/demo_schm/vol_1/sales.csv", header=True, inferSchema=True)
# df.display()
# df.printSchema
# df.schema
df.count()


In [0]:
# Step 2: Explore the Data Find the total number of rows Find the total number of columns Find distinct customers Find distinct products Generate summary statistics
# Number of rows
df.count()

# Number of columns
len(df.columns)

# Distinct customers
# df.select("customer_id").distinct().display()
# df.select("customer_id").distinct().count()

# Distinct products
# df.select("product_id").distinct().count()

# Summary statistics
# df.describe().show()
# df.describe().display()



In [0]:
# Step 3: Clean the Data
# Remove duplicate rows 
# df.distinct() 
# df.dropDuplicates()

# Remove duplicate records based on transaction_id
df.dropDuplicates(["transaction_id"])

# Remove rows containing null values
df = df.dropna(subset=["order_id", "customer_id", "transaction_id","product_id"])
# df.display()




In [0]:
# step 4 tramsform the data
# Rename columns
# df = df.withColumnRenamed("discount_amount", "discounted_aamount")

df = df.withColumnRenamed("total_amount", "sales_amount") \
       .withColumnRenamed("discount_amount", "discount")



In [0]:
df = add_metaData(df)
# df.display()

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("dev.silver.cleaned_sales")
# spark.table("dev.silver.cleaned_sales").show()

In [0]:
sales_df = spark.table("dev.silver.cleaned_sales")

In [0]:
# total sales
# sales_df.select(
#     F.sum("sales_amount").alias("total_sales")
# ).show()

# via sql 
spark.sql("""
    SELECT SUM(sales_amount) as total_sales
    FROM dev.silver.cleaned_sales
""").show()

In [0]:
%sql
-- # total number of irders
SELECT COUNT(DISTINCT order_id) AS total_orders
FROM dev.silver.cleaned_sales;

-- unique customers
SELECT COUNT(DISTINCT customer_id) AS total_customers
FROM dev.silver.cleaned_sales;

-- total sales for each customer 
SELECT customer_id, SUM(sales_amount) AS total_sales
FROM dev.silver.cleaned_sales
GROUP BY customer_id
ORDER BY total_sales;

-- Calculate total sales for each product
SELECT product_id,SUM(sales_amount) AS total_sales
FROM dev.silver.cleaned_sales
GROUP BY product_id
ORDER BY total_sales DESC;

-- Display the Top 10 customers by sales.
SELECT customer_id, SUM(sales_amount) AS total_sales
FROM dev.silver.cleaned_sales
GROUP BY customer_id
ORDER BY total_sales
LIMIT 10;

-- Display the Top 10 products by sales.
SELECT product_id, SUM(sales_amount) as total_sales
FROM dev.silver.cleaned_sales
GROUP BY product_id
ORDER BY total_sales DESC
LIMIT 10;

-- Find the highest sales transaction.
SELECT MAX(sales_amount) AS highest_sales
FROM dev.silver.cleaned_sales;

-- Find the lowest sales transaction.
SELECT MIN(sales_amount) AS lowest_sales
FROM dev.silver.cleaned_sales;

-- Calculate the average order value.
SELECT AVG(sales_amount) AS avg_order_value
FROM dev.silver.cleaned_sales;



In [0]:
# df.show()
df.printSchema()